In [1]:
HUGGINGFACE_ACCESS_TOKEN=" "
OPENAI_KEY=" "

TASK = 'taxonomy-discovery'

In [3]:

from ontolearner import LearnerPipeline, train_test_split, Conference
from ontolearner.learner import (
    AutoLLMLearner,
    LLMAugmentedRetrieverLearner,
    LLMAugmentedRAGLearner,
    StandardizedPrompting,
    LabelMapper,
)

def load_term_typing_split(test_size: float = 0.2):
    ontology = load_term_typing()
    return train_test_split(ontology, test_size=test_size, random_state=42)


def load_term_typing():
    ontology = Conference()
    ontology.load()
    return ontology.extract()


def run_auto_llm():
    train_data, test_data = load_term_typing_split()
    pipeline = LearnerPipeline(
        llm_id="Qwen/Qwen3-0.6B",
        hf_token=HUGGINGFACE_ACCESS_TOKEN,
        batch_size=32,
        max_new_tokens=10,
    )
    return pipeline(train_data=train_data, test_data=test_data, task=TASK, evaluate=True)


def run_auto_retriever():
    train_data, test_data = load_term_typing_split()
    pipeline = LearnerPipeline(
        retriever_id="sentence-transformers/all-MiniLM-L6-v2",
        top_k=5,
    )
    return pipeline(train_data=train_data, test_data=test_data, task=TASK, evaluate=True)


def run_auto_rag():
    train_data, test_data = load_term_typing_split()
    pipeline = LearnerPipeline(
        retriever_id="sentence-transformers/all-MiniLM-L6-v2",
        llm_id="Qwen/Qwen3-0.6B",
        hf_token=HUGGINGFACE_ACCESS_TOKEN,
        top_k=5,
        batch_size=16,
    )
    return pipeline(train_data=train_data, test_data=test_data, task=TASK, evaluate=True)

def run_augmented_retriever():
    train_data, test_data = load_term_typing_split()
    data = load_term_typing()
    from ontolearner.learner.retriever import LLMAugmenterGenerator, LLMAugmentedRetriever
    llm_augmenter_generator = LLMAugmenterGenerator(model_id='gpt-4.1-mini', token = OPENAI_KEY, top_n_candidate=10)
    augments = {"config": llm_augmenter_generator.get_config()}
    augments[TASK] = llm_augmenter_generator.augment(data, task=TASK)
    
    retriever = LLMAugmentedRetrieverLearner(base_retriever=LLMAugmentedRetriever(), top_k=5)
    retriever.set_augmenter(augments)
    
    pipeline = LearnerPipeline(retriever=retriever, retriever_id="sentence-transformers/all-MiniLM-L6-v2")
    return pipeline(train_data=train_data, test_data=test_data, task=TASK, evaluate=True)

def run_augmented_rag():
    train_data, test_data = load_term_typing_split()
    data = load_term_typing()
    from ontolearner.learner.retriever import LLMAugmenterGenerator, LLMAugmentedRetriever
    llm_augmenter_generator = LLMAugmenterGenerator(model_id='gpt-4.1-mini', token = OPENAI_KEY, top_n_candidate=10)
    augments = {"config": llm_augmenter_generator.get_config()}
    augments[TASK] = llm_augmenter_generator.augment(data, task=TASK)
    
    retriever = LLMAugmentedRetrieverLearner(base_retriever=LLMAugmentedRetriever(), top_k=5)

    
    llm = AutoLLMLearner(
        prompting=StandardizedPrompting,
        label_mapper=LabelMapper(),
        token=HUGGINGFACE_ACCESS_TOKEN,
        batch_size=16,
        max_new_tokens=10,
    )
    rag = LLMAugmentedRAGLearner(retriever=retriever, llm=llm)
    rag.set_augmenter(augments)
    
    pipeline = LearnerPipeline(
        rag=rag,
        retriever_id="sentence-transformers/all-MiniLM-L6-v2",
        llm_id="Qwen/Qwen3-0.6B",
        ontologizer_data=True,
    )
    return pipeline(train_data=train_data, test_data=test_data, task=TASK, evaluate=True)

2026-03-30 14:52:33.687210: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-30 14:52:33.764010: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-30 14:52:42.196519: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Failed to initialize

In [4]:
outputs = run_auto_llm()
print("Metrics:", outputs.get("metrics"))
print("Elapsed time:", outputs["elapsed_time"])

`torch_dtype` is deprecated! Use `dtype` instead!
/nfs/home/babaeih/onto-leaarner/ontolearner/learner/llm.py:102: UserWarning: No requirement for fiting the taxonomy-discovery model, the predict module will use the input data to do the 'is-a' relationship detection
  warnings.warn("No requirement for fiting the taxonomy-discovery model, the predict module will use the input data to do the 'is-a' relationship detection")
100%|██████████| 3/3 [00:13<00:00,  4.40s/it]

Metrics: {'f1_score': 0.042105263157894736, 'precision': 0.023255813953488372, 'recall': 0.2222222222222222, 'total_correct': 2, 'total_predicted': 86, 'total_ground_truth': 9}
Elapsed time: 13.206007719039917


In [5]:
outputs = run_auto_retriever()
print("Metrics:", outputs.get("metrics"))
print("Elapsed time:", outputs["elapsed_time"])

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
/nfs/home/babaeih/onto-leaarner/ontolearner/learner/retriever/learner.py:80: UserWarning: No requirement for fiting the taxonomy discovery model, the predict module will use the input data to do the fit as well.
  warnings.warn("No requirement for fiting the taxonomy discovery model, the predict module will use the input data to do the fit as well.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Metrics: {'f1_score': 0.12631578947368421, 'precision': 0.06976744186046512, 'recall': 0.6666666666666666, 'total_correct': 6, 'total_predicted': 86, 'total_ground_truth': 9}
Elapsed time: 0.13821125030517578


In [6]:
outputs = run_auto_rag()
print("Metrics:", outputs.get("metrics"))
print("Elapsed time:", outputs["elapsed_time"])

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
/nfs/home/babaeih/onto-leaarner/ontolearner/learner/rag/rag.py:68: UserWarning: No requirement for fiting the taxonomy discovery model, the predict module will use the input data to do the fit as well.
  warnings.warn("No requirement for fiting the taxonomy discovery model, the predict module will use the input data to do the fit as well.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:14<00:00,  2.33s/it]

Metrics: {'f1_score': 0.1348314606741573, 'precision': 0.075, 'recall': 0.6666666666666666, 'total_correct': 6, 'total_predicted': 80, 'total_ground_truth': 9}
Elapsed time: 14.042728662490845


In [8]:
outputs = run_augmented_retriever()
print("Metrics:", outputs.get("metrics"))
print("Elapsed time:", outputs["elapsed_time"])

33it [00:52,  1.60s/it]
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
/nfs/home/babaeih/onto-leaarner/ontolearner/learner/retriever/learner.py:80: UserWarning: No requirement for fiting the taxonomy discovery model, the predict module will use the input data to do the fit as well.
  warnings.warn("No requirement for fiting the taxonomy discovery model, the predict module will use the input data to do the fit as well.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Metrics: {'f1_score': 0.09815950920245399, 'precision': 0.05194805194805195, 'recall': 0.8888888888888888, 'total_correct': 8, 'total_predicted': 154, 'total_ground_truth': 9}
Elapsed time: 0.05592513084411621


In [10]:
outputs = run_augmented_rag()
print("Metrics:", outputs.get("metrics"))
print("Elapsed time:", outputs["elapsed_time"])

33it [00:40,  1.23s/it]
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
/nfs/home/babaeih/onto-leaarner/ontolearner/learner/rag/rag.py:68: UserWarning: No requirement for fiting the taxonomy discovery model, the predict module will use the input data to do the fit as well.
  warnings.warn("No requirement for fiting the taxonomy discovery model, the predict module will use the input data to do the fit as well.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:22<00:00,  2.26s/it]

Metrics: {'f1_score': 0.10596026490066225, 'precision': 0.056338028169014086, 'recall': 0.8888888888888888, 'total_correct': 8, 'total_predicted': 142, 'total_ground_truth': 9}
Elapsed time: 22.675782442092896
